# v34 Unified One-Run All-Routes Sweep

Single notebook version. Run once from top to bottom.

It tries, in order:
1. CPU textual/Horn-like rule sweep over v32.2 baseline.
2. Optional GPU Qwen3-8B BASE advocate generation, if CUDA and model path are available.
3. CPU rerank/select over generated base advocates.
4. Master selector comparing all routes against locked v32.2.

Locked baseline: `v32.2`, macro-F1 `0.596858548901435`.

Safety: every selected route must reload saved preds, recompute metrics, keep MC frozen, keep Unknown non-degraded, avoid No collapse, and beat strict gate `baseline + 0.01`. Otherwise it rolls back to v32.2.

In [ ]:
import json, re, math, os, sys, random, itertools, statistics, traceback, time
from pathlib import Path
from collections import Counter, defaultdict

# ================================
# User-editable config
# ================================
RUN_GPU_BASE_ADVOCATE = 'auto'  # 'auto', True, or False
BASE_MODEL_OVERRIDE = None      # e.g. '/kaggle/input/qwen3-8b/transformers/default/1'
MAX_NEW_TOKENS = 420
TEMPERATURE = 0.2
TOP_P = 0.9
RANDOM_SEED = 42

BASELINE_MACRO = 0.596858548901435
STRICT_GAIN = 0.01
BANKED = {14, 71, 109, 125}
LABELS = ['A','B','C','D','Yes','No','Unknown']

CANDIDATE_DIRS = [
    Path('/kaggle/working'),
    Path('/kaggle/input/datasets/yixuanisthebest/v2333333'),
    Path('/kaggle/input/datasets/nguyenminhtric/test-pipeline'),
    Path('/kaggle/input'),
    Path('/mnt/data'),
]

random.seed(RANDOM_SEED)

def find_file(names, required=True):
    if isinstance(names, str): names = [names]
    # direct search
    for d in CANDIDATE_DIRS:
        for name in names:
            p = d / name
            if p.exists(): return p
    # recursive search
    for d in CANDIDATE_DIRS:
        if d.exists():
            for root, dirs, files in os.walk(d):
                for name in names:
                    if name in files:
                        return Path(root)/name
    if required:
        raise FileNotFoundError(f'Could not find any of {names} in {CANDIDATE_DIRS}')
    return None

def load_json(names, required=True):
    p = find_file(names, required=required)
    if p is None: return None, None
    with open(p, 'r', encoding='utf-8') as f:
        return json.load(f), p

def save_json(obj, name):
    p = Path('/kaggle/working')/name
    p.parent.mkdir(parents=True, exist_ok=True)
    with open(p, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    print('saved', p)
    return p

def prf_for_label(rows, lab):
    tp=sum(1 for r in rows if r.get('gold')==lab and r.get('pred')==lab)
    fp=sum(1 for r in rows if r.get('gold')!=lab and r.get('pred')==lab)
    fn=sum(1 for r in rows if r.get('gold')==lab and r.get('pred')!=lab)
    prec=tp/(tp+fp) if tp+fp else 0.0
    rec=tp/(tp+fn) if tp+fn else 0.0
    f1=2*prec*rec/(prec+rec) if prec+rec else 0.0
    return {'precision':prec,'recall':rec,'f1':f1,'support':tp+fn,'pred_count':tp+fp}

def metrics(rows):
    n=len(rows)
    acc=sum(1 for r in rows if r.get('gold')==r.get('pred'))/n
    per={lab: prf_for_label(rows, lab) for lab in LABELS}
    macro=sum(per[lab]['f1'] for lab in LABELS)/len(LABELS)
    weighted=sum(per[lab]['f1']*per[lab]['support'] for lab in LABELS)/sum(per[lab]['support'] for lab in LABELS)
    mc=[r for r in rows if r.get('is_mc')]
    ynu=[r for r in rows if not r.get('is_mc')]
    mc_macro=sum(prf_for_label(mc, lab)['f1'] for lab in ['A','B','C','D'])/4 if mc else 0
    ynu_macro=sum(prf_for_label(ynu, lab)['f1'] for lab in ['Yes','No','Unknown'])/3 if ynu else 0
    return {'n':n,'acc':acc,'macro_f1':macro,'weighted_f1':weighted,'mc_macro':mc_macro,'ynu_macro':ynu_macro,'per_label':per,
            'gold_count':dict(Counter(r.get('gold') for r in rows)), 'pred_count':dict(Counter(r.get('pred') for r in rows))}

def diffs(a,b):
    assert len(a)==len(b), (len(a), len(b))
    return [r1['idx'] for r1,r2 in zip(a,b) if r1.get('pred')!=r2.get('pred')]

def clone_rows(rows):
    return [dict(r) for r in rows]

def load_baselines():
    v27,p27 = load_json('v27_standard_preds.json', required=False)
    base,pb = load_json(['v32_2_full_preds.json','v32_2_independent_full_preds.json','v32_2_independent_standard_preds.json'], required=True)
    bm=metrics(base)
    return v27,p27,base,pb,bm

def route_passes_gates(m, base_m):
    per=m['per_label']; bper=base_m['per_label']
    return {
        'strict_gain': m['macro_f1'] > base_m['macro_f1'] + STRICT_GAIN,
        'mc_frozen': abs(m['mc_macro']-base_m['mc_macro']) < 1e-12,
        'unknown_ok': per['Unknown']['f1'] >= bper['Unknown']['f1'] - 1e-12,
        'no_ok': per['No']['f1'] >= bper['No']['f1'] - 0.03,
        'no_collapse': per['No']['pred_count'] >= 30,
        'yes_not_exploded': per['Yes']['pred_count'] <= 45,
    }

def gates_all(g):
    return all(g.values())

v27, p27, base, pb, base_m = load_baselines()
print('Loaded baseline:', pb)
print('v32.2 baseline metrics:', {k:base_m[k] for k in ['acc','macro_f1','weighted_f1','mc_macro','ynu_macro']})
assert abs(base_m['macro_f1'] - BASELINE_MACRO) < 1e-9, f'Expected v32.2 macro {BASELINE_MACRO}, got {base_m["macro_f1"]}'
if v27:
    print('diffs vs v27:', diffs(v27, base))
print('Setup OK')

## Route 1 — CPU textual/Horn-like sweep

This intentionally tries many variants but only selects if strict master gates pass. Based on v33 experience, most textual rules are expected to rollback.

In [ ]:
def extract_features(r):
    q = str(r.get('question',''))
    ex = str(r.get('explanation',''))
    txt = (q + '\n' + ex).lower()
    body = re.sub(r'final answer\s*[:\-]?\s*(yes|no|unknown|[abcd])\b.*', '', ex, flags=re.I|re.S).lower()
    support = re.findall(r'(?:premise|premises|supporting premises)\s*[:\[]?\s*([0-9,\s]+)', ex, flags=re.I)
    support_nums = set()
    for s in support:
        for n in re.findall(r'\d+', s): support_nums.add(int(n))
    final_m = re.findall(r'Final Answer\s*[:\-]?\s*(Yes|No|Unknown|[ABCD])\b', ex, flags=re.I)
    old_final = final_m[-1].title() if final_m else None
    if old_final in ['A','B','C','D']: old_final = old_final.upper()
    horn_cues = sum(txt.count(w) for w in ['if ', 'then', 'implies', 'imply', 'leads to', 'from premise', 'therefore', 'thus', 'hence'])
    affirm_cues = sum(txt.count(w) for w in ['therefore', 'thus', 'qualifies', 'can ', 'is true', 'the statement is true', 'does follow', 'complete pathway', 'all ', 'every ', 'exists'])
    neg_window = body[-500:]
    negative_near = bool(re.search(r'false|not true|cannot|does not follow|insufficient|unknown|uncertain|not supported|fails to|denying|disqualifying|unlikely|not all', neg_window))
    body_true = bool(re.search(r'(statement is true|therefore[^.]{0,140}(true|qualif|can|follow|validated|optimized|complete|meets)|thus[^.]{0,140}(true|qualif|can|follow|validated|optimized|complete|meets))', body))
    direct_positive = bool(re.search(r'(premise \d+ (states|says)[^.]{0,200}(qualif|can|is true|allows|completed|every|all|exists|therefore))', body))
    contradiction_marker = bool(re.search(r'(therefore|thus)[^.]{0,200}(true|qualifies|can|follow)[^.]{0,400}final answer\s*[:\-]?\s*no', ex, flags=re.I|re.S))
    return dict(support_count=len(support_nums), old_inner_final=old_final, horn_cues=horn_cues,
                affirm_cues=affirm_cues, negative_near=negative_near, body_true=body_true,
                direct_positive=direct_positive, contradiction_marker=contradiction_marker)

def cpu_candidate_indices(rows, params):
    out=[]
    for r in rows:
        if r.get('idx') in BANKED: continue
        if r.get('is_mc'): continue
        if r.get('pred') != 'No': continue
        f=extract_features(r)
        ok=True
        if f['support_count'] < params.get('min_support',0): ok=False
        if f['horn_cues'] < params.get('min_horn',0): ok=False
        if f['affirm_cues'] < params.get('min_affirm',0): ok=False
        if params.get('require_body_true') and not f['body_true']: ok=False
        if params.get('require_direct_positive') and not f['direct_positive']: ok=False
        if params.get('block_negative_near') and f['negative_near']: ok=False
        if params.get('require_old_no') and f['old_inner_final'] != 'No': ok=False
        if params.get('require_contradiction_marker') and not f['contradiction_marker']: ok=False
        if ok: out.append(r['idx'])
    return out

def apply_no_to_yes(rows, idxs, tag):
    new=clone_rows(rows); s=set(idxs)
    for r in new:
        if r['idx'] in s:
            r['pred']='Yes'; r['repair']=tag
    return new

param_grid=[]
for min_support in [1,2,3,4,5,6]:
  for min_horn in [1,2,3,4,5,6]:
    for min_affirm in [1,2,3,4,5,6]:
      for require_body_true in [False, True]:
        for require_direct_positive in [False, True]:
          for block_negative_near in [True, False]:
            for require_old_no in [True]:
              for require_contradiction_marker in [False, True]:
                param_grid.append(dict(min_support=min_support,min_horn=min_horn,min_affirm=min_affirm,
                                       require_body_true=require_body_true, require_direct_positive=require_direct_positive,
                                       block_negative_near=block_negative_near, require_old_no=require_old_no,
                                       require_contradiction_marker=require_contradiction_marker))
leader=[]
for k,params in enumerate(param_grid):
    idxs=cpu_candidate_indices(base, params)
    if not idxs: continue
    cand=apply_no_to_yes(base, idxs, f'v34_unified_cpu_sweep_{k}')
    m=metrics(cand); per=m['per_label']; gates=route_passes_gates(m, base_m)
    leader.append({'variant':f'cpu_sweep_{k}', 'params':params, 'n_flips':len(idxs), 'flips':idxs,
                   'macro_f1':m['macro_f1'], 'acc':m['acc'], 'weighted_f1':m['weighted_f1'], 'ynu_macro':m['ynu_macro'], 'mc_macro':m['mc_macro'],
                   'yes_f1':per['Yes']['f1'], 'no_f1':per['No']['f1'], 'unknown_f1':per['Unknown']['f1'],
                   'yes_pred_count':per['Yes']['pred_count'], 'no_pred_count':per['No']['pred_count'],
                   'delta_macro':m['macro_f1']-base_m['macro_f1'], 'gates':gates, 'passes_gates':gates_all(gates),
                   'posthoc_gold':dict(Counter(r['gold'] for r in base if r['idx'] in set(idxs)))})
leader=sorted(leader, key=lambda x:x['macro_f1'], reverse=True)
print('CPU variants tried:', len(leader))
for row in leader[:20]:
    print(row['variant'], 'macro', round(row['macro_f1'],6), 'delta', round(row['delta_macro'],6), 'n', row['n_flips'], 'gold', row['posthoc_gold'], 'passes', row['passes_gates'])
save_json({'version':'v34_unified_cpu_rule_sweep_leaderboard','baseline_macro':base_m['macro_f1'],'n_variants':len(leader),'leaderboard':leader}, 'v34_unified_cpu_rule_sweep_leaderboard.json')

cpu_selected = next((x for x in leader if x['passes_gates']), None)
if cpu_selected:
    cpu_preds=apply_no_to_yes(base, cpu_selected['flips'], 'v34_unified_cpu_selected')
    cpu_m=metrics(cpu_preds)
    save_json(cpu_preds, 'v34_unified_cpu_best_preds.json')
    reload,_=load_json('v34_unified_cpu_best_preds.json')
    assert abs(metrics(reload)['macro_f1']-cpu_m['macro_f1'])<1e-12
    cpu_summary={'version':'v34_unified_cpu_best','selection_status':'SELECT_V34_CPU','selected':cpu_selected,'metrics':cpu_m,
                 'flipped_indices':cpu_selected['flips'],'baseline_v32_2':base_m}
else:
    cpu_preds=clone_rows(base); cpu_m=base_m
    save_json(cpu_preds, 'v34_unified_cpu_best_preds.json')
    cpu_summary={'version':'v34_unified_cpu_best','selection_status':'ROLLBACK_V32_2','reason':'no CPU sweep variant passed strict gates',
                 'metrics':base_m,'baseline_v32_2':base_m,'best_candidate':leader[0] if leader else None}
save_json(cpu_summary, 'v34_unified_cpu_best_summary.json')
print('CPU route status:', cpu_summary['selection_status'], cpu_summary['metrics']['macro_f1'])

## Route 2 — Optional GPU Qwen3-8B BASE advocate generation

This section runs only if GPU and a base model path are available. It uses BASE model only, no LoRA/PEFT. If no GPU/model is found, it writes a skipped route and continues.

In [ ]:
def should_run_gpu():
    if RUN_GPU_BASE_ADVOCATE is True: return True
    if RUN_GPU_BASE_ADVOCATE is False: return False
    try:
        import torch
        return torch.cuda.is_available()
    except Exception:
        return False

def find_base_model_dir():
    if BASE_MODEL_OVERRIDE:
        p=Path(BASE_MODEL_OVERRIDE)
        if p.exists(): return p
    # Common Kaggle paths or recursive config search.
    candidates=[]
    for root in [Path('/kaggle/input'), Path('/mnt/data')]:
        if root.exists():
            for dirpath, dirnames, filenames in os.walk(root):
                if 'config.json' in filenames:
                    low=str(dirpath).lower()
                    if 'qwen' in low and ('8b' in low or 'qwen3' in low):
                        candidates.append(Path(dirpath))
    # prefer non-lora/base-looking paths
    candidates=sorted(candidates, key=lambda p: (('lora' in str(p).lower()) or ('adapter' in str(p).lower()), len(str(p))))
    return candidates[0] if candidates else None

def prompt_for(row, target):
    q=str(row.get('question',''))
    ex=str(row.get('explanation',''))[:2400]
    return f"""You are a strict logic verifier for an educational QA task.\n\nQuestion:\n{q}\n\nCurrent selected explanation, which may contain an inconsistent final answer:\n{ex}\n\nTask: Evaluate whether the target answer {target} is logically supported.\nReturn exactly this format:\nReasoning: <brief premise-grounded reasoning>\nSupporting Premises: [numbers if available, else []]\nFinal Answer: {target}\nVERDICT: VALID or INVALID\n\nRules:\n- VALID only if the target answer is entailed by the premises/context.\n- INVALID if it is contradicted or not sufficiently supported.\n- Do not copy the old final answer if it conflicts with the target.\n"""

FINAL_RE = re.compile(r'Final Answer\s*[:\-]?\s*(Yes|No|Unknown|[ABCD])\b', re.I)
VERDICT_RE = re.compile(r'VERDICT\s*[:\-]?\s*(VALID|INVALID)\b', re.I)

def parse_advocate_text(text):
    finals=FINAL_RE.findall(text or '')
    verdicts=VERDICT_RE.findall(text or '')
    final=finals[-1].title() if finals else None
    if final in ['A','B','C','D']: final=final.upper()
    verdict=verdicts[-1].upper() if verdicts else None
    cited=bool(re.search(r'Supporting Premises\s*[:\-]?\s*\[[^\]]*\d', text or '', flags=re.I))
    return {'text':text, 'final_answer':final, 'verdict':verdict, 'cited':cited}

def generate_base_advocates():
    meta={'route':'base_advocate_generation','load_error':None,'model_path':None,'n_generated':0,'n_skipped':0,'skipped':[], 'target_indices':[]}
    advocates={'__meta__':meta}
    targets=[r for r in base if (not r.get('is_mc')) and r.get('pred')=='No' and r.get('idx') not in BANKED]
    meta['target_indices']=[r['idx'] for r in targets]
    if not should_run_gpu():
        meta['load_error']='GPU disabled or unavailable; skipped base advocate generation.'
        meta['n_skipped']=len(targets); meta['skipped']=meta['target_indices']
        return advocates
    model_dir=find_base_model_dir()
    if not model_dir:
        meta['load_error']='Could not find Qwen base model directory with config.json under /kaggle/input or /mnt/data.'
        meta['n_skipped']=len(targets); meta['skipped']=meta['target_indices']
        return advocates
    meta['model_path']=str(model_dir)
    try:
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
        tok=AutoTokenizer.from_pretrained(str(model_dir), trust_remote_code=True)
        try:
            model=AutoModelForCausalLM.from_pretrained(str(model_dir), dtype=dtype, device_map='auto', trust_remote_code=True)
        except TypeError:
            model=AutoModelForCausalLM.from_pretrained(str(model_dir), torch_dtype=dtype, device_map='auto', trust_remote_code=True)
        model.eval()
        for row in targets:
            pack={}
            for target in ['Yes','No','Unknown']:
                user_prompt=prompt_for(row,target)
                messages=[{'role':'user','content':user_prompt}]
                try:
                    inp=tok.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors='pt', enable_thinking=False)
                except TypeError:
                    try:
                        inp=tok.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors='pt')
                    except Exception:
                        inp=tok(user_prompt, return_tensors='pt').input_ids
                if hasattr(inp, 'to'):
                    inp=inp.to(model.device)
                with torch.no_grad():
                    out=model.generate(inp, max_new_tokens=MAX_NEW_TOKENS, do_sample=True, temperature=TEMPERATURE, top_p=TOP_P, pad_token_id=tok.eos_token_id)
                gen=out[0][inp.shape[-1]:]
                text=tok.decode(gen, skip_special_tokens=True)
                pack[target]=parse_advocate_text(text)
            advocates[str(row['idx'])]=pack
            meta['n_generated']+=1
            if meta['n_generated'] % 5 == 0:
                print('generated', meta['n_generated'], '/', len(targets))
        # cleanup
        try:
            del model
            import gc; gc.collect(); torch.cuda.empty_cache()
        except Exception: pass
    except Exception as e:
        meta['load_error']=repr(e)+'\n'+traceback.format_exc()[-2000:]
        # if partial, keep partial candidates
        existing=[int(k) for k in advocates.keys() if str(k).isdigit()]
        done=set(existing)
        meta['skipped']=[r['idx'] for r in targets if r['idx'] not in done]
        meta['n_skipped']=len(meta['skipped'])
    meta['n_skipped']=len([r for r in targets if str(r['idx']) not in advocates])
    meta['skipped']=[r['idx'] for r in targets if str(r['idx']) not in advocates]
    return advocates

advocates=generate_base_advocates()
save_json(advocates, 'v34_unified_base_advocate_candidates.json')
print('Base advocate meta:', advocates.get('__meta__'))

## Route 2b — Rerank/select base advocates

Runs even if generation skipped; in that case it simply rolls back.

In [ ]:
def valid_adv(a):
    return isinstance(a,dict) and a.get('verdict')=='VALID' and a.get('cited') and a.get('final_answer') in ['Yes','No','Unknown']

def invalid_adv(a):
    return isinstance(a,dict) and a.get('verdict')=='INVALID'

adv=advocates
variants=[]
variant_defs=[
    ('S1_unique_valid_yes_no_unknown_invalid', lambda pack: valid_adv(pack.get('Yes')) and pack.get('Yes',{}).get('final_answer')=='Yes' and invalid_adv(pack.get('No')) and invalid_adv(pack.get('Unknown'))),
    ('S2_yes_valid_no_invalid', lambda pack: valid_adv(pack.get('Yes')) and pack.get('Yes',{}).get('final_answer')=='Yes' and invalid_adv(pack.get('No'))),
    ('S3_yes_valid_no_not_valid_unknown_not_valid', lambda pack: valid_adv(pack.get('Yes')) and pack.get('Yes',{}).get('final_answer')=='Yes' and not valid_adv(pack.get('No')) and not valid_adv(pack.get('Unknown'))),
]
for name, cond in variant_defs:
    flips=[]
    for r in base:
        if r.get('idx') in BANKED or r.get('is_mc') or r.get('pred')!='No': continue
        pack=adv.get(str(r['idx']), {}) if isinstance(adv,dict) else {}
        try:
            if cond(pack): flips.append(r['idx'])
        except Exception:
            pass
    cand=apply_no_to_yes(base, flips, 'v34_unified_base_advocate_'+name)
    m=metrics(cand); gates=route_passes_gates(m, base_m)
    variants.append({'name':name,'flips':flips,'n_flips':len(flips),'metrics':m,'delta_macro':m['macro_f1']-base_m['macro_f1'],
                     'gates':gates, 'passes_gates':gates_all(gates),
                     'posthoc_gold':dict(Counter(r['gold'] for r in base if r['idx'] in set(flips)))})
variants=sorted(variants, key=lambda x:x['metrics']['macro_f1'], reverse=True)
for v in variants:
    print(v['name'], 'macro', round(v['metrics']['macro_f1'],6), 'delta', round(v['delta_macro'],6), 'n', v['n_flips'], 'gold', v['posthoc_gold'], 'passes', v['passes_gates'], 'flips', v['flips'])
save_json({'version':'v34_unified_base_advocate_rerank_leaderboard','baseline_v32_2':base_m,'advocate_meta':adv.get('__meta__') if isinstance(adv,dict) else None,'variants':variants}, 'v34_unified_base_advocate_rerank_leaderboard.json')

base_adv_selected=next((v for v in variants if v['passes_gates']), None)
if base_adv_selected:
    adv_preds=apply_no_to_yes(base, base_adv_selected['flips'], 'v34_unified_base_advocate_selected_'+base_adv_selected['name'])
    adv_m=metrics(adv_preds)
    save_json(adv_preds, 'v34_unified_base_advocate_full_preds.json')
    reload,_=load_json('v34_unified_base_advocate_full_preds.json')
    assert abs(metrics(reload)['macro_f1']-adv_m['macro_f1'])<1e-12
    adv_summary={'version':'v34_unified_base_advocate_full','selection_status':'SELECT_V34_BASE_ADVOCATE','selected':base_adv_selected,'metrics':adv_m,'baseline_v32_2':base_m,'flipped_indices':base_adv_selected['flips']}
else:
    adv_preds=clone_rows(base); adv_m=base_m
    save_json(adv_preds, 'v34_unified_base_advocate_full_preds.json')
    adv_summary={'version':'v34_unified_base_advocate_full','selection_status':'ROLLBACK_V32_2','reason':'no advocate variant passed strict gates or generation skipped','metrics':base_m,'baseline_v32_2':base_m,'best_candidate':variants[0] if variants else None,'advocate_meta':adv.get('__meta__') if isinstance(adv,dict) else None}
save_json(adv_summary, 'v34_unified_base_advocate_full_summary.json')
print('Base advocate route status:', adv_summary['selection_status'], adv_summary['metrics']['macro_f1'])

## Master selector

Compares CPU and base-advocate routes, then writes final artifacts. If no route passes strict gates, it writes v32.2 baseline as final rollback.

In [ ]:
routes=[]
for route_name, summary, preds in [
    ('cpu', cpu_summary, cpu_preds),
    ('base_advocate', adv_summary, adv_preds),
]:
    m=metrics(preds); gates=route_passes_gates(m, base_m)
    routes.append({'route':route_name, 'selection_status':summary.get('selection_status'), 'metrics':m,
                   'delta_macro':m['macro_f1']-base_m['macro_f1'], 'diffs_from_v32_2':diffs(base,preds),
                   'gates':gates, 'passes_gates':gates_all(gates), 'preds':preds, 'summary':summary})
print('Routes:')
for r in routes:
    print(r['route'], r['selection_status'], 'macro', r['metrics']['macro_f1'], 'delta', r['delta_macro'], 'diffs', r['diffs_from_v32_2'], 'passes', r['passes_gates'])

qualified=sorted([r for r in routes if r['passes_gates']], key=lambda x:x['metrics']['macro_f1'], reverse=True)
if qualified:
    best=qualified[0]
    final_preds=best['preds']; final_m=metrics(final_preds)
    save_json(final_preds, 'v34_unified_final_preds.json')
    reload,_=load_json('v34_unified_final_preds.json')
    rm=metrics(reload)
    assert abs(rm['macro_f1']-final_m['macro_f1'])<1e-12
    final_summary={'version':'v34_unified_master_final','selection_status':'SELECT_V34','selected_route':best['route'],
                   'metrics':rm,'baseline_v32_2':base_m,'diffs_from_v32_2':diffs(base,reload),
                   'all_routes':[{k:v for k,v in r.items() if k not in ['preds']} for r in routes]}
else:
    final_preds=clone_rows(base)
    save_json(final_preds, 'v34_unified_final_preds.json')
    reload,_=load_json('v34_unified_final_preds.json')
    rm=metrics(reload)
    assert abs(rm['macro_f1']-base_m['macro_f1'])<1e-12
    final_summary={'version':'v34_unified_master_final','selection_status':'ROLLBACK_V32_2','reason':'no route passed strict master gates',
                   'metrics':rm,'baseline_v32_2':base_m,'diffs_from_v32_2':diffs(base,reload),
                   'all_routes':[{k:v for k,v in r.items() if k not in ['preds']} for r in routes]}
save_json(final_summary, 'v34_unified_final_summary.json')

risk={'version':'v34_unified_risk_report','final_decision':final_summary['selection_status'],
      'metrics':final_summary['metrics'],'delta_vs_v32_2':{k:final_summary['metrics'][k]-base_m[k] for k in ['acc','macro_f1','weighted_f1','mc_macro','ynu_macro']},
      'n_flips_from_v32_2':len(final_summary.get('diffs_from_v32_2', [])),
      'flipped_indices_from_v32_2':final_summary.get('diffs_from_v32_2', []),
      'overfit_risk':'LOW_ROLLBACK' if final_summary['selection_status'].startswith('ROLLBACK') else 'MEDIUM_SWEEP_SELECTED_REQUIRES_MANUAL_AUDIT',
      'underfit_risk':'STILL_HIGH_YES_NO_REMAINING',
      'artifact_risk':'LOW_RELOADED_SAVED_PREDS',
      'label_collapse':not route_passes_gates(final_summary['metrics'], base_m)['no_collapse'] or not route_passes_gates(final_summary['metrics'], base_m)['yes_not_exploded'],
      'routes':[{k:v for k,v in r.items() if k not in ['preds']} for r in routes]}
save_json(risk, 'v34_unified_risk_report.json')
print('\nFINAL:', final_summary['selection_status'], final_summary['metrics'])
print('ALL V34 UNIFIED ASSERTS PASSED')